# Lesson 11 (Project 03): Multi-Session RAG Continuity & Pass-State Gating

In this project notebook, you will:
1. Implement Document Chunking, Metadata Extraction, Indexing Status, and Citation-Based Q&A.
2. Ground answers with **Google AI Studio Free Tier** (`gemini-2.5-flash`) + offline fallback.
3. Run the Pass-State Gate verifier before updating `feature_list.json`.

In [ ]:
!pip install -q -U google-genai

In [ ]:
import os

GEMINI_API_KEY = None
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

client = None
if GEMINI_API_KEY:
    from google import genai
    client = genai.Client(api_key=GEMINI_API_KEY)

chunks = [{"id": "arch#0", "text": "Keep AGENTS.md concise and store deep docs in docs/."}]

def ask_grounded(query: str) -> str:
    context = "\n".join(f"[{c['id']}] {c['text']}" for c in chunks)
    if client:
        resp = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=f"Answer using only these sources and cite [id]:\n{context}\nQ: {query}",
        )
        return resp.text.strip()
    return f"Keep AGENTS.md concise and store deep docs in docs/. [{chunks[0]['id']}]"

answer = ask_grounded("Where should deep docs live?")
print(answer)
assert "[arch#0]" in answer